In [ ]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report


In [ ]:
DATA_PATH = "/content/drive/MyDrive/road_acc/US_Accidents_March23.csv"

df = pd.read_csv(DATA_PATH, nrows=500000)  # sample for Colab safety
print(df.shape)
df.head()


(500000, 46)


,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-1,Source2,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.865147,-84.058723,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Night
1,A-2,Source2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.928059,-82.831184,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Night,Night,Day
2,A-3,Source2,2,2016-02-08 06:49:27,2016-02-08 07:19:27,39.063148,-84.032608,NaN,NaN,0.01,...,False,False,False,False,True,False,Night,Night,Day,Day
3,A-4,Source2,3,2016-02-08 07:23:34,2016-02-08 07:53:34,39.747753,-84.205582,NaN,NaN,0.01,...,False,False,False,False,False,False,Night,Day,Day,Day
4,A-5,Source2,2,2016-02-08 07:39:07,2016-02-08 08:09:07,39.627781,-84.188354,NaN,NaN,0.01,...,False,False,False,False,True,False,Day,Day,Day,Day


In [ ]:
def preprocess_data_v2(df):
    df = df.dropna(subset=[
        "Start_Lat", "Start_Lng", "Start_Time", "Severity"
    ])

    df["Start_Time"] = pd.to_datetime(df["Start_Time"])

    # Time features
    df["Hour"] = df["Start_Time"].dt.hour
    df["Is_Night"] = (df["Sunrise_Sunset"] == "Night").astype(int)

    # Safety: fill missing booleans
    bool_cols = [
        "Traffic_Signal", "Junction", "Crossing", "Stop",
        "Traffic_Calming", "Roundabout"
    ]
    for col in bool_cols:
        df[col] = df[col].fillna(False).astype(int)

    # Visibility
    df["Visibility(mi)"] = df["Visibility(mi)"].fillna(
        df["Visibility(mi)"].median()
    )

    # Rain indicator
    df["Is_Rain"] = df["Weather_Condition"].fillna("").str.contains(
        "Rain", case=False
    ).astype(int)

    return df


In [ ]:
df = preprocess_data_v2(df)


In [ ]:
def create_grids(df, grid_size=0.02):
    df["Lat_Grid"] = (df["Start_Lat"] // grid_size) * grid_size
    df["Lng_Grid"] = (df["Start_Lng"] // grid_size) * grid_size
    return df


In [ ]:
df = create_grids(df)


In [ ]:
grid = (
    df.groupby(["Lat_Grid", "Lng_Grid"])
      .agg(
          # Severity & frequency (ONLY for labeling, not prediction)
          Accident_Count=("Severity", "count"),
          Avg_Severity=("Severity", "mean"),
          Night_Accidents=("Is_Night", "sum"),

          # Infrastructure
          Signal_Count=("Traffic_Signal", "sum"),
          Junction_Count=("Junction", "sum"),
          Crossing_Count=("Crossing", "sum"),
          Stop_Count=("Stop", "sum"),
          Traffic_Calming_Count=("Traffic_Calming", "sum"),
          Roundabout_Count=("Roundabout", "sum"),

          # Environment
          Avg_Visibility=("Visibility(mi)", "mean"),
          Rain_Accidents=("Is_Rain", "sum"),

          # Time
          Avg_Hour=("Hour", "mean")
      )
      .reset_index()
)

grid.head()


,Lat_Grid,Lng_Grid,Accident_Count,Avg_Severity,Night_Accidents,Signal_Count,Junction_Count,Crossing_Count,Stop_Count,Traffic_Calming_Count,Roundabout_Count,Avg_Visibility,Rain_Accidents,Avg_Hour
0,25.42,-80.52,1,2.0,0,0,0,0,0,0,0,10.000000,0,8.000000
1,25.42,-80.48,4,2.0,1,0,0,0,0,0,0,10.000000,0,10.750000
2,25.44,-80.48,13,2.0,2,4,0,7,0,0,0,9.692308,2,13.769231
3,25.44,-80.46,3,2.0,2,1,0,2,1,0,0,9.666667,0,7.333333
4,25.44,-80.44,1,2.0,1,0,0,0,0,0,0,9.000000,0,6.000000


In [ ]:
grid["Risk_Score"] = (
    grid["Accident_Count"] * 0.5 +
    grid["Avg_Severity"] * 2 +
    grid["Night_Accidents"] * 0.3
)


In [ ]:
high_th = grid["Risk_Score"].quantile(0.75)
mid_th  = grid["Risk_Score"].quantile(0.40)

def risk_label(score):
    if score >= high_th:
        return "High"
    elif score >= mid_th:
        return "Medium"
    else:
        return "Low"

grid["Risk_Level"] = grid["Risk_Score"].apply(risk_label)
grid["Risk_Level"].value_counts()


,count
Risk_Level,
Low,13359
Medium,12033
High,8466


In [ ]:
feature_cols = [
    "Signal_Count",
    "Junction_Count",
    "Crossing_Count",
    "Stop_Count",
    "Traffic_Calming_Count",
    "Roundabout_Count",
    "Avg_Visibility",
    "Rain_Accidents",
    "Avg_Hour"
]

X = grid[feature_cols]
y = grid["Risk_Level"]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)


In [ ]:
rf_v2 = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_v2.fit(X_train, y_train)


RandomForestClassifier(max_depth=10, n_estimators=150, n_jobs=-1,
                       random_state=42)

In [ ]:
y_pred = rf_v2.predict(X_test)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

        High       0.91      0.79      0.84      2117
         Low       0.70      0.90      0.79      3340
      Medium       0.69      0.53      0.60      3008

    accuracy                           0.74      8465
   macro avg       0.76      0.74      0.74      8465
weighted avg       0.75      0.74      0.73      8465



In [ ]:
importance = pd.Series(
    rf_v2.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

importance


,0
Avg_Visibility,0.284320
Rain_Accidents,0.245995
Signal_Count,0.204120
Avg_Hour,0.088685
Junction_Count,0.081550
Crossing_Count,0.067719
Stop_Count,0.026502
Traffic_Calming_Count,0.000873
Roundabout_Count,0.000235


In [ ]:
model_dir = "/content/drive/MyDrive/road_acc/models"
os.makedirs(model_dir, exist_ok=True)

joblib.dump(rf_v2, f"{model_dir}/risk_model_v2.pkl")

print("✅ CrashSense-V2 model saved")


✅ CrashSense-V2 model saved
